# Advanced EDA and Text Mining

This notebook runs the same reusable analysis functions as the local Streamlit app. It profiles the source data, addresses common missing-value and duplicate issues, extracts issue/entity/keyword tags, clusters similar descriptions, and exports charts and a stakeholder report. The included default dataset contains public NHTSA consumer vehicle complaints for selected 2019 models. These reports are not verified defects or a representative sample of the vehicle fleet; complaint counts are not failure rates and text patterns do not establish root causes.

In [ ]:
# Import the shared analysis functions so the notebook and app stay consistent.
from pathlib import Path
import pandas as pd
from IPython.display import display

from analysis import (
    clean_dataset,
    detect_date_columns,
    detect_text_columns,
    generate_report,
    load_dataset,
    mine_text,
    monthly_trend,
    profile_columns,
)

## 1. Configure the input

The included NHTSA dataset is downloaded from the public NHTSA complaints API by `fetch_nhtsa_complaints.py` and covers selected 2019 model-year vehicles. It contains consumer-submitted narratives and basic incident attributes. Reports are not independently verified, counts are not fleet failure rates, and the sample does not represent every vehicle. For Excel files, the first sheet is loaded by default; change `SHEET_NAME` when needed.

In [ ]:
# Use the included public complaint snapshot; rerun the fetch script to refresh it.
DATA_PATH = Path(r"data\nhtsa_public_vehicle_complaints.csv")
SHEET_NAME = 0
EXAMPLE_ENTITIES_PATH = None
OUTPUT_DIR = Path("outputs")

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set DATA_PATH to an existing CSV/XLSX file: {DATA_PATH}")
data = load_dataset(DATA_PATH, DATA_PATH.name, sheet_name=SHEET_NAME)
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")
display(data.head())

## 2. Data quality and column profile

String whitespace is trimmed, common null-like strings become missing, duplicate rows are removed, categorical missing values become `Unknown`, and numeric missing values are filled with the column median when one is available. The audit preserves before/after counts so the cleanup is visible. A wholly missing numeric column cannot be median-imputed.

In [ ]:
# Clean the data and inspect its row counts, duplicates, and missing values.
cleaned, audit = clean_dataset(data)
print({
    "source_rows": audit["original_rows"],
    "rows_after_deduplication": audit["rows_after_deduplication"],
    "exact_duplicate_rows": audit["exact_duplicate_rows"],
    "duplicates_after_normalization": audit["duplicates_after_whitespace_normalization"],
    "missing_before": int(audit["missing_before_imputation"].sum()),
    "missing_after": int(audit["missing_after_imputation"].sum()),
})
profile = profile_columns(cleaned)
display(profile)

## 3. Statistical summaries and distributions

Review summary statistics and plots for numeric columns, then inspect the most frequent values in low-cardinality categorical columns. These are exploratory views; interpret the most useful columns in the context of product and service operations.

In [ ]:
# Display numeric summaries and export histograms for numeric fields.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import re

numeric_columns = list(cleaned.select_dtypes(include="number").columns)
if numeric_columns:
    display(cleaned[numeric_columns].describe().T)
    for column in numeric_columns:
        ax = cleaned[column].dropna().plot(kind="hist", title=f"Distribution: {column}", bins=20)
        ax.set_xlabel(column)
        safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", column).strip("_").lower()
        ax.figure.savefig(OUTPUT_DIR / f"distribution_{safe_name}.png", dpi=140, bbox_inches="tight")
        display(ax.figure)
else:
    print("No numeric columns were detected.")

# Plot the top 15 values for manageable categorical columns.
text_candidates = set(detect_text_columns(cleaned))
categorical_columns = [
    column for column in cleaned.columns
    if column not in numeric_columns
    and column not in text_candidates
    and 2 <= cleaned[column].nunique(dropna=True) <= 100
    and not any(token in str(column).casefold() for token in ("url", "uri", "source", "link", "path"))
]
for column in categorical_columns:
    counts = cleaned[column].value_counts(dropna=False).head(15)
    ax = counts.sort_values().plot(kind="barh", title=f"Top values: {column}")
    ax.set_xlabel("Records")
    safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", str(column)).strip("_").lower()
    ax.figure.savefig(OUTPUT_DIR / f"distribution_{safe_name}.png", dpi=140, bbox_inches="tight")
    display(ax.figure)

## 4. Free-text selection and text mining

Candidate text columns are suggested using string type, average length, and value diversity. Review the suggestions and edit `TEXT_COLUMNS` to include only the complaint/event descriptions. The optional example workbook is read as terms from its cells and supplements the generic component/entity lexicon.

In [ ]:
# Review suggested fields and adjust this list to match the actual dataset.
suggested_text_columns = detect_text_columns(cleaned)
print("Suggested free-text columns:", suggested_text_columns)
TEXT_COLUMNS = suggested_text_columns

# Load optional entity examples, if supplied.
extra_entity_terms = []
if EXAMPLE_ENTITIES_PATH is not None:
    example_path = Path(EXAMPLE_ENTITIES_PATH)
    if not example_path.is_file():
        raise FileNotFoundError(f"Entity reference workbook not found: {example_path}")
    example_sheets = pd.read_excel(example_path, sheet_name=None, header=None)
    for sheet_data in example_sheets.values():
        extra_entity_terms.extend(
            str(value).strip()
            for value in sheet_data.to_numpy().ravel()
            if pd.notna(value) and str(value).strip()
        )
    extra_entity_terms = list(dict.fromkeys(extra_entity_terms))
    print(f"Loaded {len(extra_entity_terms)} entity examples")

if not TEXT_COLUMNS:
    raise ValueError("Set TEXT_COLUMNS to one or more free-text columns before mining.")
mined, cluster_themes = mine_text(cleaned, TEXT_COLUMNS, extra_entity_terms)
display(mined[["Record_ID", *TEXT_COLUMNS, "Issue_Type", "Entity_Tags", "Keyword_Tags", "Text_Cluster", "Cluster_Top_Terms"]].head(20))
issue_counts = mined["Issue_Type"].value_counts().rename("Record_Count")
display(issue_counts.to_frame())
ax = issue_counts.sort_values().plot(kind="barh", title="Mined issue categories")
ax.set_xlabel("Records")
ax.figure.savefig(OUTPUT_DIR / "issue_categories.png", dpi=140, bbox_inches="tight")
print("Cluster representative terms:", cluster_themes)

## 5. Trends and frequency changes

A monthly trend is calculated only when a date field can be selected and parsed. Raw counts alone are not failure rates; compare them with units sold/installed or another exposure denominator when available.

In [ ]:
# Choose a date field after checking automatic suggestions.
suggested_date_columns = detect_date_columns(cleaned)
print("Suggested date columns:", suggested_date_columns)
DATE_COLUMN = suggested_date_columns[0] if suggested_date_columns else None  # Edit if necessary

if DATE_COLUMN is not None:
    trend = monthly_trend(cleaned, DATE_COLUMN)
    display(trend)
    if not trend.empty:
        ax = trend.set_index("Month")["Record_Count"].plot(
            kind="line", marker="o", title="Monthly event frequency"
        )
        ax.set_ylabel("Records")
        ax.figure.savefig(OUTPUT_DIR / "monthly_event_frequency.png", dpi=140, bbox_inches="tight")
        display(ax.figure)
else:
    trend = pd.DataFrame(columns=["Month", "Record_Count"])
    print("No date column selected; skipping time trend.")

## 6. Export stakeholder deliverables

The final cell saves the cleaned data, extracted tag table, optional monthly trend, and a stakeholder Markdown report. Recommendations are grounded in the uploaded data and explicitly call out the validation needed for suspected root causes.

In [ ]:
# Write the full tag export, monthly data and stakeholder report.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(OUTPUT_DIR / "cleaned_dataset.csv", index=False, encoding="utf-8-sig")
mined.to_csv(OUTPUT_DIR / "extracted_text_tags.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"Cluster": list(cluster_themes), "Representative_Terms": ["; ".join(values) for values in cluster_themes.values()]}).to_csv(
    OUTPUT_DIR / "cluster_themes.csv", index=False, encoding="utf-8-sig"
)
if not trend.empty:
    trend.to_csv(OUTPUT_DIR / "monthly_event_trend.csv", index=False, encoding="utf-8-sig")

report = generate_report(
    data,
    cleaned,
    audit,
    mined_data=mined,
    representatives=cluster_themes,
    date_column=DATE_COLUMN,
)
(OUTPUT_DIR / "stakeholder_report.md").write_text(report, encoding="utf-8")
print(f"Saved analysis outputs under: {OUTPUT_DIR.resolve()}")
print(report)